In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data_estaciones")
NDVI_PATH = Path("ndvi_semanal_estaciones_2021_2025.csv")


def build_weekly_station(df: pd.DataFrame, station: str) -> pd.DataFrame:
    d = df.copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    d = d.dropna(subset=["date"]).copy()

    # Semana ISO con inicio en lunes, igual que el NDVI semanal
    d["semana"] = d["date"].dt.to_period("W-MON").dt.start_time

    # Mantener solo columnas numéricas
    exclude = {"date"}
    numeric_cols = [
        col for col in d.columns
        if col not in exclude and pd.api.types.is_numeric_dtype(d[col])
    ]

    weekly = (
        d.groupby("semana", as_index=False)[numeric_cols]
        .median(numeric_only=True)
        .sort_values("semana")
        .reset_index(drop=True)
    )

    weekly["estacion"] = station
    weekly["anio"] = weekly["semana"].dt.year
    weekly = weekly[["estacion", "anio", "semana"] + numeric_cols]
    return weekly


def main():
    ne3 = pd.read_csv(DATA_DIR / "BD_NE3.csv")
    ne2 = pd.read_csv(DATA_DIR / "BD_NE2.csv")
    ndvi = pd.read_csv(NDVI_PATH)

    ndvi["semana"] = pd.to_datetime(ndvi["semana"], errors="coerce")
    ndvi = ndvi.dropna(subset=["semana"]).copy()

    for station_name, raw_df in [("NE2", ne2), ("NE3", ne3)]:
        weekly = build_weekly_station(raw_df, station_name)
        ndvi_station = ndvi[ndvi["estacion"] == station_name][["semana", "ndvi_mediana"]].copy()

        # Conserva exactamente las semanas del NDVI; el resto de columnas usa la mediana semanal
        out_all = weekly.merge(ndvi_station, on="semana", how="right").sort_values("semana").reset_index(drop=True)

        # Solo eliminamos filas sin ndvi_mediana; no borramos toda la tabla por columnas aisladas nulas
        out_no_null = out_all[out_all["ndvi_mediana"].notna()].copy().reset_index(drop=True)

        out_all_path = DATA_DIR / f"{station_name}_semanal_mediana_con_ndvi.csv"
        out_all.to_csv(out_all_path, index=False)

        out_no_null_path = DATA_DIR / f"{station_name}_semanal_mediana_con_ndvi_sin_nulos.csv"
        out_no_null.to_csv(out_no_null_path, index=False)

        print(f"Generado: {out_all_path.name} ({len(out_all)} filas)")
        print(f"Generado: {out_no_null_path.name} ({len(out_no_null)} filas)")


if __name__ == "__main__":
    main()

Generado: NE2_semanal_mediana_con_ndvi.csv (314 filas)
Generado: NE2_semanal_mediana_con_ndvi_sin_nulos.csv (0 filas)
Generado: NE3_semanal_mediana_con_ndvi.csv (262 filas)
Generado: NE3_semanal_mediana_con_ndvi_sin_nulos.csv (0 filas)
